#PARTE 1: Limpieza, Exploración y Modificaciones

1. El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).

In [0]:
%sql --Creamos una tabla temporal tb_reservas_prueba_temp
CREATE OR REPLACE TEMP VIEW tb_reservas_prueba_temp AS
SELECT * 
FROM cursodataacademy.project_hotel.tb_reservas
WHERE hotel = 'City Hotel' AND is_canceled = '1';



`Conclusion 1: ` 
Se aisló en una vista temporal el segmento de reservas canceladas de City Hotel, sin modificar la tabla original tb_reservas.
Esto permite al equipo comercial experimentar de forma rápida y sin riesgo sobre el grupo donde se concentra la pérdida de ingresos.
Al ser una vista, siempre refleja los datos actuales y se elimina al cerrar la sesión, por lo que no ocupa almacenamiento.

2. Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos los registros.

In [0]:
%sql
UPDATE cursodataacademy.project_hotel.tb_reservas_prueba
SET babies = 0;

`Conclusion 2:` Se corrigió el error del canal de captura: el campo babies quedó en 0 en los 119,390 registros de tb_reservas_prueba.
Esto elimina el falso dato de huéspedes con bebés, que podía distorsionar los análisis de perfil de cliente y de servicios familiares.
La corrección se aplicó solo sobre la tabla de prueba, por lo que la tabla original tb_reservas conserva sus datos intactos.

3. (CAST / CONCAT) Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").


In [0]:
%sql
SELECT 
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(arrival_date_day_of_month,'-', CAST(arrival_date_month AS STRING),'-', arrival_date_year) AS arrival_date_string
FROM cursodataacademy.project_hotel.tb_reservas;

`Conclusion 3:` Se creó la columna arrival_date_string, que une día, mes y año de llegada en un solo texto legible por ejemplo "1-July-2015".
Esto evita que el equipo comercial tenga que interpretar tres columnas separadas al revisar reportes o validar reservas.
La transformación se hizo con un SELECT, por lo que la tabla original tb_reservas no se modificó.

4. (LIKE) Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

In [0]:
%sql
SELECT *
FROM cursodataacademy.project_hotel.tb_reservas
WHERE distribution_channel LIKE 'T%'



`Conclusion 4:` Se filtraron las reservas cuyo canal de distribución empieza con “T”, y en la muestra visible todas pertenecen al canal TA/TO.
Esto permite al equipo comercial aislar el segmento intermediado, que depende de comisiones y concentra reservas con anticipaciones muy variadas.
En esa muestra aparecen tanto reservas confirmadas como canceladas, lo que indica que el canal no garantiza por sí solo la permanencia del cliente.

5. (COUNT / SUM / AVG) Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

In [0]:
%sql
SELECT 
    COUNT(*) AS Cant_Total, 
    SUM(total_of_special_requests) AS SUM_SolicitudesEspeciales,
    AVG(adr) AS Prom_PrecioPorNoche
FROM cursodataacademy.project_hotel.tb_reservas

`Conclusion 5:` El hotel registra 119,390 reservas con un precio promedio de S/101.83 por noche.
Se acumulan 68,215 solicitudes especiales, es decir, unas 0.57 por reserva: en promedio, menos de una de cada dos reservas pide algo adicional.
Esto indica margen para ofrecer servicios extra a quienes aún no personalizan su estadía.

6. (DELETE) En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

In [0]:
%sql
DELETE FROM cursodataacademy.project_hotel.tb_reservas_prueba
WHERE adr = 0.0;

`Conclusion 6:` Se eliminaron 1,959 reservas con precio por noche igual a cero de tb_reservas_prueba, al considerarse pruebas del sistema y no reservas reales.
Esto limpia la base de registros que distorsionaban métricas como el precio promedio y los ingresos estimados.
La limpieza se aplicó solo a la tabla de prueba, por lo que tb_reservas conserva sus datos originales.

7. (GROUP BY) El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

In [0]:
%sql
SELECT 
    distribution_channel,
    COUNT(1) AS CantPorCanal
FROM cursodataacademy.project_hotel.tb_reservas
GROUP BY distribution_channel

`Conclusion 7:` El canal TA/TO concentra 97870 de las 119390 reservas, por lo que el hotel depende en gran medida de agencias y operadores turísticos.
Los canales Direct tiene una cantidad de 14645 y Corporate 6677 representan aproximadamente un 18%, mientras que GDS y Undefined son casi irrelevantes.
Esta concentración es un riesgo: cualquier cambio en las condiciones o comisiones de las agencias afectaría la mayoría de las reservas.

8. (CASE WHEN / Tipos de Datos) Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium"

In [0]:
%sql --IF()
SELECT 
    adr,
    CASE
        WHEN adr < 50 THEN 'Economico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estandar'
        ELSE 'Premium'
    END AS Clasificacion_adr
FROM cursodataacademy.project_hotel.tb_reservas


`Conclusion 8:` Se clasificaron las reservas por precio en tres segmentos: Económico (menos de 50), Estándar (50 a 150) y Premium (más de 150).
En la muestra visible, la mayoría cae en Estándar (precios entre 75 y 145), con muy pocas Premium y solo reservas en 0 como Económico.
Esto permite al equipo comercial diseñar ofertas distintas por segmento y detectar dónde está el ingreso por noche más alto.

9. (Funciones de Fecha - DATE_SUB / DATE_ADD) (Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

In [0]:
%sql
SELECT
    arrival_date,
    lead_time,
    DATE_ADD(arrival_date, -CAST(lead_time AS INT)) AS Fecha_Reservacion
FROM cursodataacademy.project_hotel.tb_reservas

`Conclusion 9:` Se calculó la fecha en que se hizo cada reserva restando el lead_time a la fecha de llegada usando DATE_ADD y un valor negativo.
En la muestra visible, la mayoría reserva con semanas o pocos meses de anticipación.
Esto permite al equipo comercial ver desde cuándo se reserva, planificar campañas de venta anticipada y proyectar la ocupación futura.

10. (Funciones Matemáticas) Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.

In [0]:
%sql
SELECT 
    lead_time,
    CEIL(lead_time/30) AS meses_anticipacion_techo
FROM cursodataacademy.project_hotel.tb_reservas

`Conclusion 10:` Se convirtió el lead_time a meses de anticipación redondeados hacia arriba, para medir de forma simple cuánto tiempo antes reserva cada cliente.
En la muestra visible, la mayoría reserva con 1 a 3 meses de anticipación, aunque hay casos extremos de 12 y 25 meses.
Esto permite al equipo comercial agrupar reservas por plazo y asignar un nivel de riesgo, ya que a mayor anticipación suele haber más oportunidad de cancelar.